In [ ]:
# limit the number of threads on clusters, by Chao, 02/06/2025
import sys, os
os.environ['OMP_NUM_THREADS'] = '10'
os.environ['OPENBLAS_NUM_THREADS'] = '10'
os.environ['MKL_NUM_THREADS'] = '10'
os.environ['VECLIB_MAXIMUM_THREADS'] = '10'
os.environ['NUMEXPR_NUM_THREADS'] = '10'

import dolfin as dl
import pandas as pd
import numpy as np
import utils as ut

from scipy.interpolate import griddata, interp1d

In [ ]:
# Import GNSS data
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"
# obs_disp_name = "Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_disp_name = "Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed
# note that the height is in m, Dt in years, the original velocity data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
data = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                          'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                          'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
data['lon'] = -1*data['lon'] # convert to east positive, as the original data is west positive

azimuth = 45  # trench azimuth in degrees

# Get the normal and parallel components along the azimuth direction
data['v_trnorm'], data['v_trpara'] = ut.project_vector_2d_matrix(data['vx_Car'], data['vy_Car'], azimuth)

# When including the trench-parallel component, we removed 11 mm/yr of northwestward block translation
# (Figure 1) that was observed across all stations southwest of the Costa Rican volcanic chain 
# [Feng et al., 2012], since this motion does not correspond to elastic behavior along the megathrust 
# interface.
# This means, all stations except 2, ACOS, VERA
v_const_para = 11     # only remove the a constant value from trench parallel component 
mask1 = ~data['site'].isin(['ACOS', 'VERA'])   # stations north of the volcanic chain are not affected
data.loc[mask1, 'v_trpara'] = data.loc[mask1, 'v_trpara'] - v_const_para

#set these 2 stations' parallel component to 0 as well
data.loc[~mask1, 'v_trpara'] = 0

#For 3 sites, LOCA, BIJA, AGUS, the residuals are large, so don't use their parallel components
mask2 = data['site'].isin(['LOCA', 'BIJA', 'AGUS'])
data.loc[mask2, 'v_trpara'] = 0

#rotate back to N and E
data['vx_Car'], data['vy_Car'] = ut.project_vector_2d_matrix(data['v_trnorm'], data['v_trpara'], -azimuth)

# Convert mm to m, needed for inversion
cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                   'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
data[cols_to_convert] = data[cols_to_convert] / 1e3  # Convert mm to m

# define the centroid of relative coordinates, must be consistent with the mesh!
lon0, lat0 = -85.5+360, 10

# convert to relative locations in km
data['x'], data['y'] = ut.azi_equidist_proj(data['lon'], data['lat'], lon0, lat0)
data['z'] = 0.0

data.head()
print("Number of stations:", len(data))

# a catalog Holocene volcanoes
volc_file = "GVP_Holocene_Volcano_loc.csv" 
volc = pd.read_csv(datadir + volc_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# truncate within a region, same as Figure 1b in Feng et al 2012
volc = volc[ (volc['lat'] >= 8) & (volc['lat'] <= 12) & (volc['lon'] >= -88) & (volc['lon'] <= -83) ]
volc['x'], volc['y'] = ut.azi_equidist_proj(volc['lon'], volc['lat'], lon0, lat0)
cols_to_convert = ['x', 'y']
volc[cols_to_convert] = volc[cols_to_convert] * 1e3  # Convert km to m
volc['z'] = 0.0
# Show first few rows
# print(volc.head())

In [ ]:
# read in the 3D velocity model
veldir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
vel3dfile = "DeShon2006_3Dmodel.csv"
vel3d = pd.read_csv(veldir + vel3dfile, sep=",")
# convert to relative locations in km in my own coordinate system
vel3d['x'], vel3d['y'] = ut.azi_equidist_proj(vel3d['lon'], vel3d['lat'], lon0, lat0)
cols_to_convert = ['x', 'y', 'z']
vel3d[cols_to_convert] = vel3d[cols_to_convert] * 1e3  # Convert km to m
vel3d['z'] = vel3d['z'] * -1  # negative depth means downward
vel3d = vel3d[(vel3d['z'] <= 0)].reset_index(drop=True)  # ignore everything above the ground
print(vel3d.shape)
print(vel3d.head())

# read in the reference 1D velocity model with same depth layers
vel1dfile = "DeShon2006_1Dmodel.csv"
vel1d = pd.read_csv(veldir + vel1dfile, sep=r'\s+',skiprows=1, \
                 names=['z', 'vp', 'vs', 'vp_vs_ratio'])
vel1d['z'] = vel1d['z'] * -1 * 1e3 # negative depth means downward, Convert km to m
vel1d = vel1d[(vel1d['z'] <= 0)].reset_index(drop=True)  # ignore everything above the ground
print(vel1d)

# # read in a 1D velocity model with density, the last layer is from Kaban et al. 2016 JGR, only took depth and density
# den1dfile = "DeShon2004_1Dmodel.csv"
# den1d = pd.read_csv(veldir + den1dfile, sep=r'\s+',skiprows=1, \
#                  names=['z', 'vp', 'vp_vs_ratio', 'den'])
# den1d['vs'] = den1d['vp'] / den1d['vp_vs_ratio']
# den1d['z'] = den1d['z'] * -1 * 1e3 # negative depth means downward, Convert km to m
# den1d['den'] = den1d['den'] * 1e3 # Convert g/cm^3 to kg/m^3
# # print(den1d.head())

# read a made-up 1D velocity model of density, ref. DeShon2004_1Dmodel, but with same depth layers as 3d models
den1dfile = "Density_1Dmodel.csv"
den1d = pd.read_csv(veldir + den1dfile, sep=r'\s+',skiprows=1, \
                 names=['z', 'den'])
den1d['z'] = den1d['z'] * -1 * 1e3 # negative depth means downward, Convert km to m
den1d = den1d[(den1d['z'] <= 0)].reset_index(drop=True)  # ignore everything above the ground
den1d['den'] = den1d['den'] * 1e3 # Convert g/cm^3 to kg/m^3
# print(den1d.head())
print(den1d)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import meshio

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Given x and y coordinates
x = np.array([-100.0, 0.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 
              100.0, 110.0, 120.0, 140.0, 160.0, 180.0, 200.0, 220.0, 350.0])
y = np.array([-100.0, 0.0, 20.0, 40.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 
              120.0, 130.0, 140.0, 150.0, 160.0, 180.0, 200.0, 220.0, 350.0])

# Create grid of points
X, Y = np.meshgrid(-x, y)

# Scatter plot
ax = axes[0]
ax.scatter(X, Y, s=10, color='blue', marker='o')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.axis('equal')  # Keep aspect ratio
ax.set_title('Grid Points Scatter Plot')
ax.grid(True)

lon1, lat1 = -(84+42.00/60), 9+12.00/60
lon, lat = ut.inverse_azi_equidist_proj(X.ravel(), Y.ravel(), lon1, lat1)


# nicoya plate interface geometry from Slab2 
plate_file = "/home/staff/chao/SSEinv/Nicoya/plateinterface/nicoya_slab2_100_10_10.txt"
df = ut.parse_plate_interface_file(plate_file)

ax = axes[1]
ax.scatter(df['lon']-360, df['lat'], s=10, color='gray', marker='o')  # solid dots
# ax.scatter(lon, lat, s=10, color='blue', marker='o')  # solid dots
ax.scatter(vel3d['lon'], vel3d['lat'], s=10, color='blue', marker='o')  # solid dots
ax.scatter(data['lon'], data['lat'], s=10, color='red', marker='^')  # solid dots
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.axis("equal")  # equal scaling for x/y
ax.grid(True)


plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 5))
plt.step(vel1d['vp'], -vel1d['z']/1e3, where='mid', linewidth=1, color='blue', 
         linestyle='-', label='Vp DeShon2006')
# plt.step(den1d['vp'], -den1d['z']/1e3, where='mid', linewidth=1, color='red', 
#          linestyle='-', label='Vp DeShon2004')
plt.step(vel1d['vs'], -vel1d['z']/1e3, where='mid', linewidth=1, color='blue', 
         linestyle='--', label='Vs DeShon2006')
# plt.step(den1d['vs'], -den1d['z']/1e3, where='mid', linewidth=1, color='red', 
#          linestyle='--', label='Vs DeShon2004')
plt.xlabel('Velocity (km/s)')
plt.ylabel('Depth (km)')
plt.ylim([0, 200])
plt.gca().invert_yaxis()  # Depth increases downward
plt.grid(True, alpha=0.3)
plt.legend(loc='best', frameon=True, shadow=True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
import dolfin as dl

def get_layered_1d_value(mesh_depths, model_depths, model_values):
    """
    Get values from layered 1D model.
    
    Layer definition:
    - Layer i: model_depths[i-1] < z <= model_depths[i] (for i > 0)
    - Layer 0: z <= model_depths[0] 
    - Last layer: z <= model_depths[-1] (extends to negative infinity)
    
    Example with model_depths = [0, -5000, -9000]:
    - Layer 0: z <= 0 → use model_values[0]
    - Layer 1: -5000 < z <= 0 → use model_values[1] 
    - Layer 2: z <= -5000 → use model_values[2]
    
    Parameters:
    -----------
    mesh_depths : array
        Depth values at mesh nodes (all negative)
    model_depths : array
        Depth values from 1D model (sorted shallow to deep: 0, -5000, -9000, ...)
    model_values : array
        Property values from 1D model
        
    Returns:
    --------
    result : array
        Property values at mesh depths using layered lookup
    """
    result = np.zeros_like(mesh_depths, dtype=model_values.dtype)

    for i, depth in enumerate(mesh_depths):
        # Find which layer this depth falls into
        # We want the shallowest layer depth that is >= mesh depth
        
        layer_idx = len(model_values) - 1  # Default to deepest layer

        if depth <= model_depths[-1]:
            layer_idx = layer_idx  # Use deepest layer

        else:    
            for j in range(len(model_depths) - 1):  # Avoid index out of bounds
                if depth <= model_depths[j] and depth > model_depths[j+1]:
                    layer_idx = j
                    break        

        result[i] = model_values[layer_idx]

    return result


def process_velocity_models(vel3d, vel1d, den1d, mesh, verbose=True):
    """
    Process 3D and 1D velocity models with density and project to FEniCS mesh.
    
    1D models are treated as layered (constant values within each layer).
    3D model uses interpolation between grid points.
    
    Parameters:
    -----------
    vel3d : pd.DataFrame
        3D velocity model with columns: x, y, z (m), vs (km/s)
    vel1d : pd.DataFrame  
        1D velocity model with columns: z (m), vs (km/s)
    den1d : pd.DataFrame
        1D density model with columns: z (m), den (kg/m³)
    mesh : dolfin.Mesh
        FEniCS mesh object
    verbose : bool, default=True
        If True, print detailed progress information
    
    Returns:
    --------
    vs_function : dolfin.Function
        Shear velocity field on mesh (km/s)
    density_function : dolfin.Function  
        Density field on mesh (kg/m³)
    shear_modulus_function : dolfin.Function
        Shear modulus field on mesh (GPa)
    CG_mu : dolfin.FunctionSpace
        Function space used
    """
    
    if verbose:
        print("Starting velocity model processing...")
    
    # Step 1: Clean the data - remove any NaN values
    vel3d_clean = vel3d.dropna(subset=['x', 'y', 'z', 'vs']).copy()
    vel1d_clean = vel1d.dropna(subset=['z', 'vs']).copy()
    den1d_clean = den1d.dropna(subset=['z', 'den']).copy()
    
    if verbose:
        print(f"Data after cleaning:")
        print(f"  3D model: {len(vel3d_clean)} points")
        print(f"  1D velocity: {len(vel1d_clean)} layers")
        print(f"  1D density: {len(den1d_clean)} layers")
    
    # Check if we have enough data
    if len(vel3d_clean) == 0:
        raise ValueError("No valid data points in 3D model after removing NaN values")
    if len(vel1d_clean) == 0:
        raise ValueError("No valid data points in 1D velocity model after removing NaN values")
    if len(den1d_clean) == 0:
        raise ValueError("No valid data points in 1D density model after removing NaN values")
    
    # Step 2: Sort 1D models by depth (shallow to deep: 0, -5000, -9000, ...)
    # This is CRITICAL for the layered model to work correctly
    vel1d_clean = vel1d_clean.sort_values('z', ascending=False).reset_index(drop=True)
    den1d_clean = den1d_clean.sort_values('z', ascending=False).reset_index(drop=True)
    
    if verbose:
        print(f"1D models sorted by depth (shallow to deep):")
        print(f"  Velocity depths: {vel1d_clean['z'].values[:5]} ... {vel1d_clean['z'].values[-2:]}")
        print(f"  Density depths: {den1d_clean['z'].values[:5]} ... {den1d_clean['z'].values[-2:]}")
    
    # Step 3: Extract 3D bounding box
    x_min, x_max = vel3d_clean['x'].min(), vel3d_clean['x'].max()
    y_min, y_max = vel3d_clean['y'].min(), vel3d_clean['y'].max()  
    z_min, z_max = vel3d_clean['z'].min(), vel3d_clean['z'].max()
    
    if verbose:
        print(f"3D model bounds:")
        print(f"  X: {x_min:.1f} to {x_max:.1f} m")
        print(f"  Y: {y_min:.1f} to {y_max:.1f} m") 
        print(f"  Z: {z_min:.1f} to {z_max:.1f} m")
    
    # Step 4: Get mesh node coordinates
    mesh_coords = mesh.coordinates()
    n_nodes = len(mesh_coords)
    
    if verbose:
        print(f"Mesh has {n_nodes} nodes")
        print(f"Mesh depth range: {mesh_coords[:, 2].min():.1f} to {mesh_coords[:, 2].max():.1f} m")
        print(f"1D model depth range: {vel1d_clean['z'].min():.1f} to {vel1d_clean['z'].max():.1f} m")
        
        # Check if mesh extends beyond 1D model
        if mesh_coords[:, 2].min() < vel1d_clean['z'].min():
            deepest_vs = vel1d_clean['vs'].iloc[-1]
            deepest_den = den1d_clean['den'].iloc[-1]
            print(f"Note: Mesh extends {vel1d_clean['z'].min() - mesh_coords[:, 2].min():.1f} m beyond deepest 1D layer")
            print(f"      These nodes will use deepest layer values (vs={deepest_vs:.3f} km/s, density={deepest_den:.1f} kg/m³)")
    
    # Step 5: Determine which nodes are inside 3D model domain
    inside_3d = ((mesh_coords[:, 0] >= x_min) & (mesh_coords[:, 0] <= x_max) &
                 (mesh_coords[:, 1] >= y_min) & (mesh_coords[:, 1] <= y_max) &
                 (mesh_coords[:, 2] >= z_min) & (mesh_coords[:, 2] <= z_max))
    
    n_inside = np.sum(inside_3d)
    n_outside = np.sum(~inside_3d)
    
    if verbose:
        print(f"Nodes inside 3D domain: {n_inside}")
        print(f"Nodes outside 3D domain: {n_outside}")
    
    # Step 6: Create FEniCS functions first (needed for DOF coordinates)
    if verbose:
        print("Creating FEniCS functions...")
    
    CG_mu = dl.FunctionSpace(mesh, "CG", 1)
    
    vs_function = dl.Function(CG_mu)
    density_function = dl.Function(CG_mu)
    shear_modulus_function = dl.Function(CG_mu)
    
    # Get DOF coordinates (these are the actual function evaluation points)
    V_coords = CG_mu.tabulate_dof_coordinates()
    n_dofs = len(V_coords)
    
    if verbose:
        print(f"Function space has {n_dofs} DOFs")
        print(f"DOF depth range: {V_coords[:, 2].min():.1f} to {V_coords[:, 2].max():.1f} m")
    
    # Step 7: Apply 1D layered models to ALL DOFs (background)
    if verbose:
        print("Applying 1D layered models to all DOFs...")
    
    vs_at_dofs = get_layered_1d_value(
        V_coords[:, 2], 
        vel1d_clean['z'].values, 
        vel1d_clean['vs'].values
    )
    
    density_at_dofs = get_layered_1d_value(
        V_coords[:, 2], 
        den1d_clean['z'].values, 
        den1d_clean['den'].values
    )
    
    if verbose:
        print(f"  1D velocity range: {vs_at_dofs.min():.3f} to {vs_at_dofs.max():.3f} km/s")
        print(f"  1D density range: {density_at_dofs.min():.1f} to {density_at_dofs.max():.1f} kg/m³")
    
    # Step 8: Override with 3D model where available
    # Find which DOFs are inside 3D domain
    inside_3d_dofs = ((V_coords[:, 0] >= x_min) & (V_coords[:, 0] <= x_max) &
                      (V_coords[:, 1] >= y_min) & (V_coords[:, 1] <= y_max) &
                      (V_coords[:, 2] >= z_min) & (V_coords[:, 2] <= z_max))
    
    n_inside_dofs = np.sum(inside_3d_dofs)
    n_outside_dofs = np.sum(~inside_3d_dofs)
    
    if verbose:
        print(f"DOFs inside 3D domain: {n_inside_dofs}")
        print(f"DOFs outside 3D domain: {n_outside_dofs}")
    
    if n_inside_dofs > 0:
        if verbose:
            print("Overriding with 3D interpolated model...")
        
        # Interpolate 3D velocity using griddata
        vs_3d = griddata(
            points=vel3d_clean[['x', 'y', 'z']].values,
            values=vel3d_clean['vs'].values,
            xi=V_coords[inside_3d_dofs],
            method='linear',
            fill_value=np.nan
        )
        
        # Handle any NaN values with nearest neighbor
        nan_mask = np.isnan(vs_3d)
        if nan_mask.any():
            if verbose:
                print(f"    Filling {np.sum(nan_mask)} NaN values with nearest neighbor")
            else:
                print(f"Warning: Filled {np.sum(nan_mask)} NaN values with nearest neighbor")
            
            vs_3d[nan_mask] = griddata(
                points=vel3d_clean[['x', 'y', 'z']].values,
                values=vel3d_clean['vs'].values,
                xi=V_coords[inside_3d_dofs][nan_mask],
                method='nearest'
            )
        
        # Override 1D values with 3D interpolated values
        vs_at_dofs[inside_3d_dofs] = vs_3d
        # Density remains 1D layered everywhere (depth-dependent only)
        
        if verbose:
            print(f"  3D velocity range: {vs_at_dofs[inside_3d_dofs].min():.3f} to {vs_at_dofs[inside_3d_dofs].max():.3f} km/s")
    
    # Step 9: Assign values to FEniCS functions
    vs_function.vector()[:] = vs_at_dofs
    density_function.vector()[:] = density_at_dofs
    
    # Step 10: Debug verification (if verbose)
    if verbose:
        print("\nDebug: Verifying function values at test points...")
        test_coords = [
            [-1000, 0, -5000],
            [-20000, 0, -15000], 
            [-20000, 10000, -28000],
            [20000, -10000, -100000]
        ]
        
        for coord in test_coords:
            try:
                density_val = density_function(*coord)
                vs_val = vs_function(*coord)
                print(f"  Point {coord}: density = {density_val:.1f} kg/m³, vs = {vs_val:.3f} km/s")
            except:
                print(f"  Point {coord}: outside mesh domain")
        
        # Check layered consistency by depth ranges
        print("\nDebug: Checking density consistency by depth ranges...")
        depth_ranges = [(-5000, 0), (-15000, -10000), (-30000, -25000), (-100000, -50000)]
        
        for z_min, z_max in depth_ranges:
            mask = (V_coords[:, 2] >= z_min) & (V_coords[:, 2] < z_max)
            if np.any(mask):
                densities_in_range = density_at_dofs[mask]
                velocities_in_range = vs_at_dofs[mask]
                print(f"  Depth {z_min} to {z_max} m:")
                print(f"    Density: {densities_in_range.min():.1f} to {densities_in_range.max():.1f} kg/m³")
                print(f"    Velocity: {velocities_in_range.min():.3f} to {velocities_in_range.max():.3f} km/s")
    
    # Step 11: Compute shear modulus (μ = ρ × vs²)
    if verbose:
        print("Computing shear modulus field...")
    
    vs_ms = vs_at_dofs * 1000.0  # km/s to m/s
    shear_modulus_pa = density_at_dofs * vs_ms**2  # kg/m³ × (m/s)² = Pa
    shear_modulus_gpa = shear_modulus_pa / 1e9  # Pa to GPa
    
    shear_modulus_function.vector()[:] = shear_modulus_gpa
    
    # Step 12: Summary
    if verbose:
        print("\nProcessing complete! Summary:")
        print(f"Final velocity: {vs_at_dofs.min():.3f} to {vs_at_dofs.max():.3f} km/s")
        print(f"Final density: {density_at_dofs.min():.1f} to {density_at_dofs.max():.1f} kg/m³")
        print(f"Final shear modulus: {shear_modulus_gpa.min():.2f} to {shear_modulus_gpa.max():.2f} GPa")
    
    return vs_function, density_function, shear_modulus_function, CG_mu


def save_functions_to_file(vs_function, density_function, shear_modulus_function,
                          output_dir="./output/", meshname="mesh", verbose=True):
    """
    Save the computed functions to XDMF files for visualization and HDF5 for storage.
    
    Parameters:
    -----------
    vs_function, density_function, shear_modulus_function : dolfin.Function
        The computed field functions
    output_dir : str
        Directory to save the files
    meshname : str
        Mesh name for file naming
    verbose : bool, default=True
        If True, print progress information
    """
    
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    if verbose:
        print(f"Saving functions to {output_dir}")
    
    # Save as XDMF files (for visualization in ParaView/VisIt)
    # Velocity
    vs_filename = output_dir + f'vs_{meshname}.xdmf'
    vs_file = dl.XDMFFile(vs_filename)
    vs_function.rename('shear velocity', 'shear velocity')
    vs_file.write(vs_function)
    vs_file.close()
    
    # Density  
    density_filename = output_dir + f'density_{meshname}.xdmf'
    density_file = dl.XDMFFile(density_filename)
    density_function.rename('density', 'density')
    density_file.write(density_function)
    density_file.close()
    
    # Shear modulus
    mu_filename = output_dir + f'shear_modulus_{meshname}.xdmf'
    mu_file = dl.XDMFFile(mu_filename)
    shear_modulus_function.rename('shear modulus', 'shear modulus')
    mu_file.write(shear_modulus_function)
    mu_file.close()
    
    # Also save as HDF5 files (for loading back into FEniCS)
    vs_h5_file = dl.HDF5File(vs_function.function_space().mesh().mpi_comm(), 
                            output_dir + f"vs_{meshname}.h5", "w")
    vs_h5_file.write(vs_function, "vs")
    vs_h5_file.close()
    
    density_h5_file = dl.HDF5File(density_function.function_space().mesh().mpi_comm(), 
                                 output_dir + f"density_{meshname}.h5", "w")
    density_h5_file.write(density_function, "density") 
    density_h5_file.close()
    
    mu_h5_file = dl.HDF5File(shear_modulus_function.function_space().mesh().mpi_comm(), 
                            output_dir + f"shear_modulus_{meshname}.h5", "w")
    mu_h5_file.write(shear_modulus_function, "shear_modulus")
    mu_h5_file.close()
    
    if verbose:
        print("Functions saved successfully!")
        print(f"XDMF files (for visualization):")
        print(f"  {vs_filename}")
        print(f"  {density_filename}") 
        print(f"  {mu_filename}")


def test_layered_model(vel1d, den1d, test_depths, verbose=True):
    """
    Test function to verify layered model implementation.
    
    Parameters:
    -----------
    vel1d, den1d : pd.DataFrame
        1D models to test
    test_depths : array
        Depths to test
    verbose : bool
        Print detailed results
    """
    
    if verbose:
        print("Testing layered model implementation:")
        print("1D Velocity Model layers:")
        for i in range(len(vel1d)):
            if i == len(vel1d) - 1:
                # Last layer extends to negative infinity
                print(f"  Layer {i}: z ≤ {vel1d.iloc[i]['z']:.0f} m (deepest) → vs = {vel1d.iloc[i]['vs']:.3f} km/s")
            else:
                # Regular layer boundaries
                current_z = vel1d.iloc[i]['z']
                next_z = vel1d.iloc[i+1]['z']
                print(f"  Layer {i}: {next_z:.0f} m < z ≤ {current_z:.0f} m → vs = {vel1d.iloc[i]['vs']:.3f} km/s")
        
        print("\n1D Density Model layers:")
        for i in range(len(den1d)):
            if i == len(den1d) - 1:
                # Last layer extends to negative infinity
                print(f"  Layer {i}: z ≤ {den1d.iloc[i]['z']:.0f} m (deepest) → density = {den1d.iloc[i]['den']:.1f} kg/m³")
            else:
                # Regular layer boundaries
                current_z = den1d.iloc[i]['z']
                next_z = den1d.iloc[i+1]['z']
                print(f"  Layer {i}: {next_z:.0f} m < z ≤ {current_z:.0f} m → density = {den1d.iloc[i]['den']:.1f} kg/m³")
    
    # Test velocity lookup
    vs_test = get_layered_1d_value(test_depths, vel1d['z'].values, vel1d['vs'].values)
    den_test = get_layered_1d_value(test_depths, den1d['z'].values, den1d['den'].values)
    
    if verbose:
        print(f"\nTest Results:")
        for i, depth in enumerate(test_depths):
            print(f"  Depth {depth:.0f} m → vs = {vs_test[i]:.3f} km/s, density = {den_test[i]:.1f} kg/m³")
    
    return vs_test, den_test


def visualize_field_slices(function, mesh, field_name="Field", z_levels=None):
    """
    Create simple visualization of field at different depth slices.
    """
    
    try:
        import matplotlib.pyplot as plt
        
        if z_levels is None:
            mesh_coords = mesh.coordinates()
            z_min, z_max = mesh_coords[:, 2].min(), mesh_coords[:, 2].max()
            z_levels = np.linspace(z_min, z_max, 5)  # 5 slices
        
        print(f"Creating visualization slices for {field_name} at z = {z_levels}")
        print("For detailed visualization, save functions and use ParaView:")
        print("  paraview vs_meshname.xdmf density_meshname.xdmf shear_modulus_meshname.xdmf")
        
    except ImportError:
        print("Matplotlib not available. Use ParaView for visualization:")
        print("  paraview vs_meshname.xdmf density_meshname.xdmf shear_modulus_meshname.xdmf")


    # Example usage - complete workflow:
    """
    # Step 1: Process velocity models with correct layered implementation
    vs_func, den_func, mu_func, CG_mu = process_velocity_models(vel3d, vel1d, den1d, mesh, verbose=True)

    # Step 2: Test the layered model (optional)
    test_depths = np.array([-1000, -5000, -50000, -150000, -800000])  # Test various depths
    test_layered_model(vel1d, den1d, test_depths, verbose=True)

    # Step 3: Use shear modulus directly in your PDE
    mtrue_mu_fun = mu_func  # Contains shear modulus in GPa

    # Step 4: Save functions
    save_functions_to_file(vs_func, den_func, mu_func, 
                        output_dir="./velocity_fields/", 
                        meshname="nicoya2")

    # Step 5: Use in PDE workflow (no changes needed)
    pde_varf = PDEVarf(mtrue_mu_fun)
    pde = hp.PDEVariationalProblem(Vh, pde_varf, bc, bc0, is_fwd_linear=True)
    """

In [ ]:
test_depths = np.array([-1000, -5000, -50000, -150000, -800000])
test_layered_model(vel1d, den1d, test_depths, verbose=True)

In [ ]:
meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
print(meshname)

# Choose path data
meshpath = "/home/staff/chao/SSEinv/Nicoya/mesh/"
# load mesh
mesh = dl.Mesh(meshpath + meshname + '.xml')

vs_func, den_func, mu_func, _ = process_velocity_models(vel3d, vel1d, den1d, mesh, verbose=True)
mtrue_mu_fun = mu_func  # Contains shear modulus in GPa


In [ ]:
print( "Saving true shear modulus structure to .xdmf file" )
filename = 'true_mu_' + meshname + '.xdmf'
if not os.path.exists(filename):
    mu_id = dl.XDMFFile(filename)
    mu_func.rename('shear modulus', 'shear modulus')
    mu_id.write(mu_func)
print( mu_func.vector().min(), mu_func.vector().max() )

# Save true Vs and density structures if using 3D & 1D velocity models
print( "Saving true Vs structure to .xdmf file" )
filename = 'true_vs_' + meshname + '.xdmf'
if not os.path.exists(filename):
    vs_id = dl.XDMFFile(filename)
    vs_func.rename('shear velocity Vs', 'shear velocity Vs')
    vs_id.write(vs_func)
print( vs_func.vector().min(), vs_func.vector().max() )

print( "Saving true density structure to .xdmf file" )
filename = 'true_pho_' + meshname + '.xdmf'
if not os.path.exists(filename):
    den_id = dl.XDMFFile(filename)
    den_func.rename('density', 'density')
    den_id.write(den_func)
print( den_func.vector().min(), den_func.vector().max() )